# Validador Inteligente de DOI e ISSN
**Este cuaderno procesa archivos Excel para validar códigos contra bases de datos oficiales.**
Ejecuta las celdas en orden presionando el botón `Play`.

In [ ]:
!pip install -q groq pydantic aiohttp pandas openpyxl tenacity nest_asyncio

In [ ]:

# ==========================================
# CELDA 2: EL CEREBRO DE VALIDACIÓN
# Ejecuta esta celda para cargar el motor de IA
# ==========================================
import nest_asyncio
nest_asyncio.apply()
import os
import json
import re
import asyncio
import aiohttp
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any, Union
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import groq
import hashlib




class PublicationRecord(BaseModel):
    id_registro: str = Field(..., description="ID of the record in the matrix")
    tipo_publicacion: Optional[str] = Field(None, description="Type of publication (e.g., ARTICULO, LIBRO)")
    codigo_doi: Optional[str] = Field(None, description="Registered DOI code")
    codigo_issn: Optional[str] = Field(None, description="Registered ISSN code")
    titulo: Optional[str] = Field(None, description="Registered title")
    autores: Optional[str] = Field(None, description="Registered authors")
    anio_publicacion: Optional[str] = Field(None, description="Registered publication year")
    revista_editorial: Optional[str] = Field(None, description="Registered journal or publisher")

class ValidationResult(BaseModel):
    codigo: str = Field(..., description="DOI or ISSN code validated")
    publicacion_encontrada: str = Field(..., description="Details of the publication found in official sources")
    datos_registrados: str = Field(..., description="Details of the registered data")
    coincide: str = Field(..., description="Does it match? 'Sí' or 'No'")
    observaciones: str = Field(..., description="Inconsistencies, missing info, format errors, etc.")
    
class AIValidationReport(BaseModel):
    resultados: List[ValidationResult]







class MetadataFetcher:
    """
    Async Client for fetching metadata from public APIs (Crossref/OpenAlex) 
    with Exponential Backoff retries, persistent sessions and Polite Pool access.
    """
    def __init__(self):
        self.session: Optional[aiohttp.ClientSession] = None

    async def get_session(self) -> aiohttp.ClientSession:
        if self.session is None or self.session.closed:
            headers = {
                "User-Agent": "PublicationValidatorBot/1.0 (mailto:soporte@tudominio.com)"
            }
            timeout = aiohttp.ClientTimeout(total=15)
            self.session = aiohttp.ClientSession(timeout=timeout, headers=headers)
        return self.session

    async def close(self):
        if self.session and not self.session.closed:
            await self.session.close()

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type((aiohttp.ClientError, asyncio.TimeoutError))
    )
    async def fetch_doi_metadata(self, doi: str) -> Optional[Dict[str, Any]]:
        """
        Fetches metadata for a given DOI asynchronusly.
        """
        url = f"https://api.crossref.org/works/{doi}"
        session = await self.get_session()
        
        async with session.get(url) as response:
            if response.status == 200:
                data = await response.json()
                message = data.get("message", {})
                
                title = message.get("title", [""])[0] if message.get("title") else ""
                authors_list = message.get("author", [])
                
                # Para evitar exceder el límite de tokens de Groq (6000 TPM) con papers que tienen
                # miles de autores (ej. papers de física), extraemos máximo los primeros 20 autores
                # y truncamos la cadena final a 800 caracteres.
                authors_str_list = [f"{a.get('given', '')} {a.get('family', '')}".strip() for a in authors_list[:30]]
                authors = ", ".join(authors_str_list)
                if len(authors_list) > 30:
                    authors += " [y otros...]"
                    
                # Truncar título también por seguridad
                if len(title) > 800:
                    title = title[:800] + "..."
                
                issued = message.get("issued", {}).get("date-parts", [[None]])[0][0]
                year = str(issued) if issued else ""
                
                container_title = message.get("container-title", [""])[0] if message.get("container-title") else ""
                publisher = message.get("publisher", "")
                journal_or_publisher = container_title if container_title else publisher
                if len(journal_or_publisher) > 500:
                    journal_or_publisher = journal_or_publisher[:500] + "..."
                
                pub_type = message.get("type", "")
                
                return {
                    "titulo": title,
                    "autores": authors,
                    "anio_publicacion": year,
                    "revista_editorial": journal_or_publisher,
                    "tipo_publicacion": pub_type
                }
            elif response.status == 404:
                return None
            else:
                response.raise_for_status()

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10),
        retry=retry_if_exception_type((aiohttp.ClientError, asyncio.TimeoutError))
    )
    async def fetch_issn_metadata(self, issn: str) -> Optional[Dict[str, Any]]:
        """
        Fetches metadata for a given ISSN asynchronously.
        """
        url = f"https://api.crossref.org/journals/{issn}"
        session = await self.get_session()
        
        async with session.get(url) as response:
            if response.status == 200:
                data = await response.json()
                message = data.get("message", {})
                
                title = message.get("title", "")
                publisher = message.get("publisher", "")
                
                return {
                    "titulo": title,
                    "editorial": publisher,
                    "tipo_publicacion": "journal"
                }
            elif response.status == 404:
                # Fallback to OpenAlex
                openalex_url = f"https://api.openalex.org/sources/issn:{issn}"
                async with session.get(openalex_url) as oa_response:
                    if oa_response.status == 200:
                        data = await oa_response.json()
                        title = data.get("display_name", "")
                        publisher = data.get("host_organization_name", "")
                        country = data.get("country_code", "")
                        return {
                            "titulo": title,
                            "editorial": publisher,
                            "pais": country,
                            "tipo_publicacion": data.get("type", "")
                        }
                    elif oa_response.status == 404:
                        return None
                    else:
                        oa_response.raise_for_status()
            else:
                response.raise_for_status()










class AIAgent:
    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        try:
            self.client = groq.AsyncGroq(api_key=self.api_key)
        except Exception as e:
            print(f"Error initializing Groq client: {e}")
            self.client = None

    @retry(
        stop=stop_after_attempt(6),
        wait=wait_exponential(multiplier=2, min=5, max=60)
    )
    async def validate_doi(self, record: PublicationRecord, fetched_data: Optional[Dict[str, Any]]) -> ValidationResult:
        if not self.client:
            return ValidationResult(
                codigo=record.codigo_doi or "",
                publicacion_encontrada="Error de Configuración",
                datos_registrados=record.titulo or "Desconocido",
                coincide="No",
                observaciones="Fallo en la llamada a la IA: groq no está instalado o falta la API key."
            )
            
        system_prompt = (
            "Actúa como un especialista en gestión de información científica y validación bibliográfica. "
            "Tu objetivo es verificar que los metadatos registrados (título, autores, etc.) correspondan con los datos oficiales del DOI.\n"
            "1. En el campo 'datos_registrados' debes incluir EXACTAMENTE lo que el usuario ingresó (el Título y Autores). No lo alteres.\n"
            "2. En el campo 'publicacion_encontrada' debes incluir el título y autores reales extraídos de los datos oficiales.\n"
            "3. Si el usuario ingresó el Título y coincide con el oficial, evalúa 'coincide' como 'Sí' INCLUSO SI dejó los Autores en blanco o como 'Desconocido'. (Un solo campo correcto es suficiente para validar el DOI).\n"
            "4. CRÍTICO: Evalúa 'coincide' como 'No' SOLAMENTE si la información proporcionada es incorrecta/no coincide, o si AMBOS campos (Título y Autores) están completamente en blanco.\n"
            "5. En 'observaciones' explica en qué coinciden o fallan, detectando errores tipográficos, traducciones, o si falta información.\n"
            "DEBES RESPONDER ÚNICAMENTE CON UN JSON VÁLIDO QUE TENGA LA SIGUIENTE ESTRUCTURA:\n"
            '{"codigo": "string", "publicacion_encontrada": "string", "datos_registrados": "string", "coincide": "string (Sí/No)", "observaciones": "string"}'
        )
        
        user_prompt = f"""
        Datos Registrados (Ingresados por el usuario):
        - Título: {record.titulo}
        - Autores/Editorial: {record.revista_editorial}
        - DOI: {record.codigo_doi}
        
        Datos Encontrados Oficialmente (Crossref/Otros):
        {json.dumps(fetched_data, indent=2, ensure_ascii=False) if fetched_data else "No se encontró el DOI o el DOI es inválido."}
        
        Evalúa y responde SOLO en formato JSON.
        """

        response = await self.client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}
        )
        data = json.loads(response.choices[0].message.content)
        return ValidationResult(**data)

    @retry(
        stop=stop_after_attempt(6),
        wait=wait_exponential(multiplier=2, min=5, max=60)
    )
    async def validate_issn(self, record: PublicationRecord, fetched_data: Optional[Dict[str, Any]]) -> ValidationResult:
        if not self.client:
            return ValidationResult(
                codigo=record.codigo_issn or "",
                publicacion_encontrada="Error de Configuración",
                datos_registrados=record.titulo or "Desconocido",
                coincide="No",
                observaciones="Fallo en la llamada a la IA: groq no está instalado o falta la API key."
            )
            
        system_prompt = (
            "Actúa como un especialista en gestión de información científica y validación bibliográfica. "
            "Tu objetivo es verificar que los códigos ISSN registrados correspondan correctamente con la publicación académica asociada.\n"
            "REGLAS ESTRICTAS:\n"
            "1. En el campo 'datos_registrados' debes incluir EXACTAMENTE lo que el usuario ingresó para Título/Revista y para Revista/Editorial.\n"
            "2. En el campo 'publicacion_encontrada' debes incluir el título y editorial oficiales del ISSN.\n"
            "3. Si el usuario ingresó el Título/Revista y coincide con el oficial, evalúa 'coincide' como 'Sí' INCLUSO SI dejó la Revista/Editorial en blanco o como 'Desconocido'. (Un solo campo correcto es suficiente para validar el ISSN).\n"
            "4. CRÍTICO: Evalúa 'coincide' como 'No' SOLAMENTE si la información proporcionada es incorrecta/no coincide, o si AMBOS campos (Título y Editorial) están completamente en blanco.\n"
            "DEBES RESPONDER ÚNICAMENTE CON UN JSON VÁLIDO QUE TENGA LA SIGUIENTE ESTRUCTURA:\n"
            '{"codigo": "string", "publicacion_encontrada": "string", "datos_registrados": "string", "coincide": "string (Sí/No)", "observaciones": "string"}'
        )
        
        user_prompt = f"""
        Datos Registrados:
        - Título/Revista: {record.titulo}
        - Revista/Editorial (Ingresado por usuario): {record.revista_editorial}
        - ISSN: {record.codigo_issn}
        
        Datos Encontrados Oficialmente:
        {json.dumps(fetched_data, indent=2, ensure_ascii=False) if fetched_data else "No se encontró el ISSN o el formato es inválido."}
        
        Evalúa y responde SOLO en formato JSON.
        """

        response = await self.client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}
        )
        data = json.loads(response.choices[0].message.content)
        return ValidationResult(**data)










class PublicationValidator:
    """
    Main Orchestrator for validating DOIs and ISSNs.
    Combines async API fetchers, AI analysis and an in-memory cache.
    """
    def __init__(self, fetcher: MetadataFetcher = None, ai_api_key: str = None):
        self.fetcher = fetcher or MetadataFetcher()
        self.ai_manager = AIAgent(ai_api_key)
        self._cache: Dict[str, ValidationResult] = {}
        
    def _clean_code(self, code: Union[str, None]) -> str:
        return code.strip() if code else ""

    def _limpiar_codigo(self, raw: str) -> str:
        if not raw: return ""
        cleaned = str(raw).strip()
        if "doi.org/" in cleaned:
            cleaned = cleaned.split("doi.org/")[-1]
        
        # Eliminar prefijos comunes que el usuario pueda haber ingresado por error
        cleaned = re.sub(r'(?i)^doi:\s*', '', cleaned)
        cleaned = re.sub(r'(?i)^issn:\s*', '', cleaned)
        
        # Eliminar cualquier prefijo extraño que termine en ":" antes de un número
        if ":" in cleaned and not cleaned.startswith("10."):
            partes = cleaned.split(":", 1)
            if len(partes) > 1 and (partes[1].strip().startswith("10.") or re.match(r'^\d', partes[1].strip())):
                cleaned = partes[1].strip()
                
        cleaned = re.sub(r'[^a-zA-Z0-9.\-/:_;()]', '', cleaned)
        return cleaned

    def _is_valid_doi_format(self, doi: str) -> bool:
        if not doi:
            return False
        # Basic DOI format regex (starts with 10.)
        return bool(re.match(r'^10\.\d{4,9}/[-._;()/:A-Z0-9]+$', doi, re.IGNORECASE))
        
    def _is_valid_issn_format(self, issn: str) -> bool:
        if not issn:
            return False
        # Basic ISSN format (with or without hyphen)
        return bool(re.match(r'^\d{4}-?\d{3}[\dxX]$', issn, re.IGNORECASE))

    async def validate_record(self, record: Union[Dict[str, Any], PublicationRecord], ai_semaphore: asyncio.Semaphore = None) -> ValidationResult:
        if isinstance(record, dict):
            record = PublicationRecord(**record)
            
        doi_limpio = self._clean_code(record.codigo_doi)
        issn_limpio = self._clean_code(record.codigo_issn)
        
        # Generar llave de cache única para este registro que incluya los datos de entrada
        import hashlib
        record_str = f"{record.titulo}_{record.autores}_{record.revista_editorial}"
        record_hash = hashlib.md5(record_str.encode('utf-8')).hexdigest()
        cache_key = f"doi:{doi_limpio}_{record_hash}" if doi_limpio else f"issn:{issn_limpio}_{record_hash}"
        if cache_key in self._cache and not (not doi_limpio and not issn_limpio):
            # print(f"[CACHE HIT] Se obtuvo {cache_key} de la caché.")
            return self._cache[cache_key]
        
        if doi_limpio:
            if self._is_valid_doi_format(doi_limpio):
                record.codigo_doi = doi_limpio
                try:
                    official_data = await self.fetcher.fetch_doi_metadata(doi_limpio)
                except Exception as e:
                    # print(f"Error fetching DOI {doi_limpio} after retries: {e}")
                    official_data = None
                    
                try:
                    if ai_semaphore:
                        async with ai_semaphore:
                            await asyncio.sleep(2.1)
                            result = await self.ai_manager.validate_doi(record, official_data)
                    else:
                        result = await self.ai_manager.validate_doi(record, official_data)
                except tenacity.RetryError as e:
                    ex = e.last_attempt.exception()
                    err_msg = getattr(ex, 'message', str(ex))
                    result = ValidationResult(
                        codigo=doi_limpio,
                        publicacion_encontrada="Error Crítico (IA)",
                        datos_registrados=record.titulo or "Desconocido",
                        coincide="No",
                        observaciones=f"Error al validar con IA tras múltiples reintentos: {err_msg}"
                    )
                except Exception as e:
                    result = ValidationResult(
                        codigo=doi_limpio,
                        publicacion_encontrada="Error Crítico",
                        datos_registrados=record.titulo or "Desconocido",
                        coincide="No",
                        observaciones=f"Error inesperado al validar: {str(e)}"
                    )
                    
                self._cache[cache_key] = result
                return result
            else:
                return ValidationResult(
                    codigo=doi_limpio,
                    publicacion_encontrada="No se consultó",
                    datos_registrados=record.titulo or "Desconocido",
                    coincide="No",
                    observaciones="Error de formato: El DOI contiene caracteres incorrectos, espacios intermedios o no cumple con la estructura estándar (ej. 10.xxxx/yyyy)."
                )
                
        elif issn_limpio:
            if self._is_valid_issn_format(issn_limpio):
                record.codigo_issn = issn_limpio
                try:
                    official_data = await self.fetcher.fetch_issn_metadata(issn_limpio)
                except Exception as e:
                    # print(f"Error fetching ISSN {issn_limpio} after retries: {e}")
                    official_data = None
                    
                try:
                    if ai_semaphore:
                        async with ai_semaphore:
                            await asyncio.sleep(2.1)
                            result = await self.ai_manager.validate_issn(record, official_data)
                    else:
                        result = await self.ai_manager.validate_issn(record, official_data)
                except tenacity.RetryError as e:
                    ex = e.last_attempt.exception()
                    err_msg = getattr(ex, 'message', str(ex))
                    result = ValidationResult(
                        codigo=issn_limpio,
                        publicacion_encontrada="Error Crítico (IA)",
                        datos_registrados=record.titulo or "Desconocido",
                        coincide="No",
                        observaciones=f"Error al validar con IA tras múltiples reintentos: {err_msg}"
                    )
                except Exception as e:
                    result = ValidationResult(
                        codigo=issn_limpio,
                        publicacion_encontrada="Error Crítico",
                        datos_registrados=record.titulo or "Desconocido",
                        coincide="No",
                        observaciones=f"Error inesperado al validar: {str(e)}"
                    )
                    
                self._cache[cache_key] = result
                return result
            else:
                return ValidationResult(
                    codigo=issn_limpio,
                    publicacion_encontrada="No se consultó",
                    datos_registrados=record.titulo or "Desconocido",
                    coincide="No",
                    observaciones="Error de formato: El ISSN contiene caracteres incorrectos, carece de formato o no es un código válido (ej. 1234-5678)."
                )
                
        else:
            return ValidationResult(
                codigo="N/A",
                publicacion_encontrada="No se consultó",
                datos_registrados=record.titulo or "Desconocido",
                coincide="No",
                observaciones="Código inexistente o campo vacío. No se proporcionó ningún DOI o ISSN para validar."
            )

    async def validate_batch(self, records: List[Union[Dict[str, Any], PublicationRecord]]) -> AIValidationReport:
        """
        Validates a batch of records asynchronously, using an AI Semaphore to prevent Groq API Rate Limits.
        """
        # Semáforo para controlar la concurrencia de la IA (evitar 429 Rate Limit en Groq - 30 RPM)
        ai_sem = asyncio.Semaphore(1)
        
        async def wrap_validate(rec):
            res = await self.validate_record(rec, ai_semaphore=ai_sem)
            return res
                
        tasks = [wrap_validate(rec) for rec in records]
        results = await asyncio.gather(*tasks)
        
        # Cerrar conexión compartida de aiohttp de manera segura si se creó
        if hasattr(self.fetcher, 'close'):
            await self.fetcher.close()
            
        return AIValidationReport(resultados=list(results))

# =========================================================================================
# SECCIÓN DE COMENTARIOS: GUÍA DE INTEGRACIÓN PARA EL DESARROLLADOR (VERSIÓN ASÍNCRONA)
# =========================================================================================
# Este módulo está diseñado para producción: Asíncrono, con Caché en Memoria y Reintentos Exponenciales.
# - Tipado Estricto: Usa Pydantic (PublicationRecord, ValidationResult) para evitar errores.
# - Asincronía: Al usar async/await, se debe llamar desde un Event Loop de asyncio.
# 
# Ejemplo de uso e integración rápida en el sistema principal:
# 
# 
# from src.validator import PublicationValidator
# 
# async def main():
#     # 1. Instanciar el validador (inyectando la llave de la API de IA):
#     validador = PublicationValidator(ai_api_key="AQUI_SU_LLAVE_API_DE_IA")
# 
#     # 2. Pasar el registro crudo proveniente de la base de datos local:
#     registro_db = {
#         "id_registro": "123",
#         "titulo": "Estudio del Genoma",
#         "codigo_doi": "10.1000/xyz123"
#     }
# 
#     # 3. Validar de forma asíncrona (AWAIT es obligatorio):
#     resultado = await validador.validate_record(registro_db)
# 
#     # 4. Guardar o retornar como JSON al sistema:
#     json_listo = resultado.model_dump_json()
#     print(json_listo)
#
# 


In [ ]:

# ==========================================
# CELDA 3: INTERFAZ DE USUARIO Y EJECUCIÓN
# Ejecuta esta celda para subir tu Excel y validar
# ==========================================
from google.colab import files
import pandas as pd
import io
import time
import getpass

print("=== VALIDADOR DE DOI E ISSN (GOOGLE COLAB) ===")
print("Este proceso analizará los registros UNO POR UNO para evitar bloqueos por límites de uso gratuito.\n")

# 1. Pedir API Key de Groq
api_key = getpass.getpass("🔑 Ingresa tu GROQ API KEY (Oculta por seguridad) y presiona ENTER: ")
os.environ["GROQ_API_KEY"] = api_key

print("\n📂 Sube tu archivo Excel (debe tener las columnas TIPO, CODIGO, TITULO, AUTORES_EDITORIAL)")
uploaded = files.upload()

if not uploaded:
    print("❌ No se subió ningún archivo.")
else:
    # Tomar el primer archivo subido
    filename = list(uploaded.keys())[0]
    df = pd.read_excel(io.BytesIO(uploaded[filename]))
    print(f"✅ Archivo cargado: {len(df)} registros encontrados.")
    
    # Preparar registros
    registros_a_procesar = []
    for idx, row in df.iterrows():
        tipo = str(row.get("TIPO", "")).strip().upper()
        codigo = str(row.get("CODIGO", "")).strip()
        titulo = str(row.get("TITULO", "")).strip() if pd.notna(row.get("TITULO", "")) else ""
        autores = str(row.get("AUTORES_EDITORIAL", "")).strip() if pd.notna(row.get("AUTORES_EDITORIAL", "")) else ""
        
        if tipo not in ["DOI", "ISSN"]:
            tipo = "DOI" if "10." in codigo else "ISSN"
            
        registro = {
            "id_registro": f"Fila_{idx+2}",
            "tipo_publicacion": "ARTICULO" if tipo == "DOI" else "REVISTA",
            "titulo": titulo if titulo else "Desconocido",
            "autores": autores if autores else "Desconocidos",
            "anio_publicacion": "",
            "revista_editorial": autores if tipo == "ISSN" else ""
        }
        
        if tipo == "DOI":
            registro["codigo_doi"] = codigo
            registro["codigo_issn"] = ""
        else:
            registro["codigo_doi"] = ""
            registro["codigo_issn"] = codigo
            
        registros_a_procesar.append(registro)
        
    resultados_list = []
    total = len(registros_a_procesar)
    
    async def procesar_lote_seguro():
        validador = PublicationValidator()
        print("\n🚀 Iniciando validación inteligente, por favor espera...")
        
        for idx, rec in enumerate(registros_a_procesar):
            print(f"🔎 Analizando ({idx+1}/{total}): {rec.get('codigo_doi') or rec.get('codigo_issn')}...", end="")
            try:
                # PAUSA CONTROLADA PARA NO SATURAR GROQ (EVITA EL ERROR DE CONNECTION/RATE LIMIT)
                await asyncio.sleep(3.2)
                res = await validador.validate_record(rec)
                resultados_list.append(res)
                print(f" [OK: {res.coincide}]")
            except Exception as e:
                print(f" [ERROR: {str(e)}]")
                resultados_list.append(ValidationResult(
                    codigo=rec.get('codigo_doi') or rec.get('codigo_issn'),
                    publicacion_encontrada="Error",
                    datos_registrados=rec.get('titulo') or "Desconocido",
                    coincide="No",
                    observaciones=f"Fallo técnico: {str(e)}"
                ))
                
        # Cerrar conexión
        if hasattr(validador.fetcher, 'close'):
            await validador.fetcher.close()
            
    # Ejecutar proceso
    asyncio.run(procesar_lote_seguro())
    
    # Preparar Excel de salida
    print("\n💾 Generando archivo de resultados...")
    df_out = df.copy()
    df_out["COINCIDE_IA"] = [res.coincide for res in resultados_list]
    df_out["OBSERVACIONES_IA"] = [res.observaciones for res in resultados_list]
    df_out["DATOS_OFICIALES"] = [res.publicacion_encontrada for res in resultados_list]
    
    output_filename = f"Resultados_Colab_{filename}"
    df_out.to_excel(output_filename, index=False)
    
    print(f"🎉 ¡Proceso finalizado! Descargando {output_filename}...")
    files.download(output_filename)
